In [9]:
import pandas as pd
from datetime import datetime

df = pd.read_csv('result.csv', sep = ';')
df.head()

,slug,profession,salary,firm_name,city,employment,experience,raw_description,description,date_published,category
0,rukovoditel-otdela-soprovozhdeniya-vnutrenney-...,Руководитель отдела сопровождения внутренней и...,None,Санкт-Петербургская биржа,Москва,Полная занятость,None,"<div class=""b-b-1""> <p>Компания ""Санкт-Петербу...","Компания ""Санкт-Петербургская биржа"" Вам предс...",25 Марта 2026,Начальник отдела
1,java-senior-tekhnologii-otraslevoy-transformat...,Java (Senior)( ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМ...,None,ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМАЦИИ,Москва,Полная занятость,Более 6 лет,"<div class=""b-b-1""> <p>Компания ""ТЕХНОЛОГИИ ОТ...","Компания ""ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМАЦИИ""...",27 Марта 2026,Java
2,middle-aqa-spetsialist-85967322,Middle+ AQA специалист,от 140 000 до 180 000 руб.,Компания Ритейл Сервис,Барнаул,Полная занятость,None,"<div class=""b-b-1""> <p>Компания ""Компания Рите...","Компания ""Компания Ритейл Сервис"" Мы - аккреди...",26 Марта 2026,Специалист
3,tekhnolog-akrikhin-84569790,Технолог( АКРИХИН ),None,АКРИХИН,Старая Купавна,Полная занятость,None,"<div class=""b-b-1""> <p>Компания ""АКРИХИН""</p> ...","Компания ""АКРИХИН"" Обязанности: Обеспечивать ...",25 Марта 2026,Технолог
4,project-manager-game-dev-beresnev-games-84947044,Project Manager Game Dev( Beresnev Games ),None,Beresnev Games,Москва,Полная занятость,None,"<div class=""b-b-1""> <p>Компания ""Beresnev Game...","Компания ""Beresnev Games"" Мы в поиске Project ...",26 Марта 2026,Project manager


In [10]:
# правлю датасет Саши
df = df.drop(columns = ['slug'])
df['source'] = 'scrapping'
df = df.rename(columns = {'firm_name':'company_name', 'profession':'profession_full', 'category':'profession', 'description':'company_description', 'raw_description':'vacancy_description'})

# разбираюсь с датами
def parse_russian_date(date_str: str) -> datetime:
    months_ru = {
        'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
        'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
        'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
    }
    
    day_str, month_str, year_str = date_str.split()
    
    day = int(day_str)
    month = months_ru[month_str.lower()]
    year = int(year_str)
    
    return datetime(year, month, day)

df['date_published'] = df['date_published'].apply(parse_russian_date)


# разбираюсь с зарплатой - оставляю минимальную и максимальную границу
def low_and_upper_limit(x):

    if pd.isna(x):
        return None, None
    else:
        x = x.split()
        array = []
        for i in x:
            if i.isdigit():
                if i == '000':
                    new_x = prev_x + i
                    array[-1] = int(new_x)
                else:
                    prev_x = i
                    array.append(int(prev_x))
        
        if len(array) >= 2:
            return array[0], array[1]
        elif len(array) == 1:
            return array[0], None
        else:
            return None, None
        
df['payment_from'], df['payment_to'] = zip(*df['salary'].apply(low_and_upper_limit))
df['vacancy_count'] = None
df['staff_count'] = None
df['catalogues'] = None
# делаю интуитивный и адекватный порядок колонок
df = df[['profession', 'profession_full', 'vacancy_description', 'date_published', 'payment_from', 'payment_to', 'experience', 'employment', 'city', 'company_name', 'company_description', 'vacancy_count', 'staff_count', 'catalogues', 'source']]

df.head()

,profession,profession_full,vacancy_description,date_published,payment_from,payment_to,experience,employment,city,company_name,company_description,vacancy_count,staff_count,catalogues,source
0,Начальник отдела,Руководитель отдела сопровождения внутренней и...,"<div class=""b-b-1""> <p>Компания ""Санкт-Петербу...",2026-03-25,NaN,NaN,None,Полная занятость,Москва,Санкт-Петербургская биржа,"Компания ""Санкт-Петербургская биржа"" Вам предс...",None,None,None,scrapping
1,Java,Java (Senior)( ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМ...,"<div class=""b-b-1""> <p>Компания ""ТЕХНОЛОГИИ ОТ...",2026-03-27,NaN,NaN,Более 6 лет,Полная занятость,Москва,ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМАЦИИ,"Компания ""ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМАЦИИ""...",None,None,None,scrapping
2,Специалист,Middle+ AQA специалист,"<div class=""b-b-1""> <p>Компания ""Компания Рите...",2026-03-26,140000.0,180000.0,None,Полная занятость,Барнаул,Компания Ритейл Сервис,"Компания ""Компания Ритейл Сервис"" Мы - аккреди...",None,None,None,scrapping
3,Технолог,Технолог( АКРИХИН ),"<div class=""b-b-1""> <p>Компания ""АКРИХИН""</p> ...",2026-03-25,NaN,NaN,None,Полная занятость,Старая Купавна,АКРИХИН,"Компания ""АКРИХИН"" Обязанности: Обеспечивать ...",None,None,None,scrapping
4,Project manager,Project Manager Game Dev( Beresnev Games ),"<div class=""b-b-1""> <p>Компания ""Beresnev Game...",2026-03-26,NaN,NaN,None,Полная занятость,Москва,Beresnev Games,"Компания ""Beresnev Games"" Мы в поиске Project ...",None,None,None,scrapping


In [3]:
df.shape

(24110, 14)

In [12]:
# собираю 2 моих датасета в один
df_my_1 = pd.read_csv('vacancies_df_all_3.csv')
df_my_1 = df_my_1.loc[df_my_1['id_client'] != 4412294]
df_my_2 = pd.read_csv('vacancies_df_all.csv')
df_my = pd.concat([df_my_1, df_my_2])
df_my = df_my.drop_duplicates()
df_my.shape

(6884, 12)

In [13]:
# объединяю вакансии и компании с SuperJob
df_companies = pd.read_csv('chosen_companies_info.csv')
#df_my = pd.read_csv('vacancies_df_all.csv')
merged_df = pd.merge(df_my, df_companies, how = 'inner', left_on = 'id_client', right_on = 'company_id')
merged_df.head()

,vacancy_id,profession,id_client,date_published,candidat,town,experience,catalogues,payment_from,payment_to,...,company_id,company_title,company_link,description,vacancy_count,staff_count,industry_ids,industry_titles,client_logo,is_blocked
0,0,Дежурный специалист по информационной безопасн...,11714,1775739004,"ФФКУ ""Налог-Сервис"" ФНС России по ЦОД- это про...",Уфа,"{'id': 2, 'title': 'От 1 года'}","[{'id': 33, 'title': 'IT, Интернет, связь, тел...",0,0,...,11714,Филиал ФКУ «Налог-Сервис» ФНС России по ЦОД в ...,https://www.superjob.ru/clients/filial-fku-nal...,"Федеральный казённое учреждение, подведомствен...",71,100 — 500,NaN,NaN,https://public.superjob.ru/images/clients_logo...,False
1,1,Системный администратор,11714,1775725510,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",Фурманов,"{'id': 2, 'title': 'От 1 года'}","[{'id': 33, 'title': 'IT, Интернет, связь, тел...",0,36540,...,11714,Филиал ФКУ «Налог-Сервис» ФНС России по ЦОД в ...,https://www.superjob.ru/clients/filial-fku-nal...,"Федеральный казённое учреждение, подведомствен...",71,100 — 500,NaN,NaN,https://public.superjob.ru/images/clients_logo...,False
2,2,Инженер по эксплуатации зданий и сооружений,11714,1775725313,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",Истра,"{'id': 3, 'title': 'От 3 лет'}","[{'id': 1, 'title': 'Административная работа, ...",0,42000,...,11714,Филиал ФКУ «Налог-Сервис» ФНС России по ЦОД в ...,https://www.superjob.ru/clients/filial-fku-nal...,"Федеральный казённое учреждение, подведомствен...",71,100 — 500,NaN,NaN,https://public.superjob.ru/images/clients_logo...,False
3,3,Системный администратор,11714,1775725298,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",Скадовск,"{'id': 1, 'title': 'Без опыта'}","[{'id': 33, 'title': 'IT, Интернет, связь, тел...",30000,35000,...,11714,Филиал ФКУ «Налог-Сервис» ФНС России по ЦОД в ...,https://www.superjob.ru/clients/filial-fku-nal...,"Федеральный казённое учреждение, подведомствен...",71,100 — 500,NaN,NaN,https://public.superjob.ru/images/clients_logo...,False
4,4,Системный администратор,11714,1775725510,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",Владимир,"{'id': 1, 'title': 'Без опыта'}","[{'id': 33, 'title': 'IT, Интернет, связь, тел...",26100,0,...,11714,Филиал ФКУ «Налог-Сервис» ФНС России по ЦОД в ...,https://www.superjob.ru/clients/filial-fku-nal...,"Федеральный казённое учреждение, подведомствен...",71,100 — 500,NaN,NaN,https://public.superjob.ru/images/clients_logo...,False


In [15]:
#оставляю только то, что надо
merged_df['city'] = merged_df['town'].apply(lambda x: x.get('title') if isinstance(x, dict) else None)
merged_df['experience'] = merged_df['experience'].apply(lambda x: x.get('title') if isinstance(x, dict) else None)
merged_df_new = merged_df[['profession', 'date_published', 'payment_from', 'payment_to', 'experience',  'town', 'company_title', 'description', 'vacancy_count', 'staff_count', 'catalogues', 'candidat']]
merged_df_new = merged_df_new.rename(columns = {'town':'city', 'company_title':'company_name', 'description':'company_description', 'candidat':'vacancy_description'})
merged_df_new['profession_full'] = merged_df_new['profession']
merged_df_new['employment'] = None
merged_df_new['source'] = 'api'
merged_df_new = merged_df_new[['profession', 'profession_full', 'vacancy_description', 'date_published', 'payment_from', 'payment_to', 'experience', 'employment', 'city', 'company_name', 'company_description', 'vacancy_count', 'staff_count', 'catalogues', 'source']]
merged_df_new['date_published'] = merged_df_new['date_published'].apply(lambda x: pd.to_datetime(datetime.fromtimestamp(x).date()))
merged_df_new.head()


,profession,profession_full,vacancy_description,date_published,payment_from,payment_to,experience,employment,city,company_name,company_description,vacancy_count,staff_count,catalogues,source
0,Дежурный специалист по информационной безопасн...,Дежурный специалист по информационной безопасн...,"ФФКУ ""Налог-Сервис"" ФНС России по ЦОД- это про...",2026-04-09,0,0,None,None,Уфа,Филиал ФКУ «Налог-Сервис» ФНС России по ЦОД в ...,"Федеральный казённое учреждение, подведомствен...",71,100 — 500,"[{'id': 33, 'title': 'IT, Интернет, связь, тел...",api
1,Системный администратор,Системный администратор,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",2026-04-09,0,36540,None,None,Фурманов,Филиал ФКУ «Налог-Сервис» ФНС России по ЦОД в ...,"Федеральный казённое учреждение, подведомствен...",71,100 — 500,"[{'id': 33, 'title': 'IT, Интернет, связь, тел...",api
2,Инженер по эксплуатации зданий и сооружений,Инженер по эксплуатации зданий и сооружений,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",2026-04-09,0,42000,None,None,Истра,Филиал ФКУ «Налог-Сервис» ФНС России по ЦОД в ...,"Федеральный казённое учреждение, подведомствен...",71,100 — 500,"[{'id': 1, 'title': 'Административная работа, ...",api
3,Системный администратор,Системный администратор,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",2026-04-09,30000,35000,None,None,Скадовск,Филиал ФКУ «Налог-Сервис» ФНС России по ЦОД в ...,"Федеральный казённое учреждение, подведомствен...",71,100 — 500,"[{'id': 33, 'title': 'IT, Интернет, связь, тел...",api
4,Системный администратор,Системный администратор,"ФКУ ""Налог Сервис"" - это IT-компания, подведом...",2026-04-09,26100,0,None,None,Владимир,Филиал ФКУ «Налог-Сервис» ФНС России по ЦОД в ...,"Федеральный казённое учреждение, подведомствен...",71,100 — 500,"[{'id': 33, 'title': 'IT, Интернет, связь, тел...",api


In [16]:
final_df = pd.concat([df, merged_df_new])
final_df = final_df.drop_duplicates()
final_df.head()

,profession,profession_full,vacancy_description,date_published,payment_from,payment_to,experience,employment,city,company_name,company_description,vacancy_count,staff_count,catalogues,source
0,Начальник отдела,Руководитель отдела сопровождения внутренней и...,"<div class=""b-b-1""> <p>Компания ""Санкт-Петербу...",2026-03-25,NaN,NaN,None,Полная занятость,Москва,Санкт-Петербургская биржа,"Компания ""Санкт-Петербургская биржа"" Вам предс...",None,None,None,scrapping
1,Java,Java (Senior)( ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМ...,"<div class=""b-b-1""> <p>Компания ""ТЕХНОЛОГИИ ОТ...",2026-03-27,NaN,NaN,Более 6 лет,Полная занятость,Москва,ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМАЦИИ,"Компания ""ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМАЦИИ""...",None,None,None,scrapping
2,Специалист,Middle+ AQA специалист,"<div class=""b-b-1""> <p>Компания ""Компания Рите...",2026-03-26,140000.0,180000.0,None,Полная занятость,Барнаул,Компания Ритейл Сервис,"Компания ""Компания Ритейл Сервис"" Мы - аккреди...",None,None,None,scrapping
3,Технолог,Технолог( АКРИХИН ),"<div class=""b-b-1""> <p>Компания ""АКРИХИН""</p> ...",2026-03-25,NaN,NaN,None,Полная занятость,Старая Купавна,АКРИХИН,"Компания ""АКРИХИН"" Обязанности: Обеспечивать ...",None,None,None,scrapping
4,Project manager,Project Manager Game Dev( Beresnev Games ),"<div class=""b-b-1""> <p>Компания ""Beresnev Game...",2026-03-26,NaN,NaN,None,Полная занятость,Москва,Beresnev Games,"Компания ""Beresnev Games"" Мы в поиске Project ...",None,None,None,scrapping


In [17]:
final_df.shape

(30867, 15)

In [18]:
df.head()

,profession,profession_full,vacancy_description,date_published,payment_from,payment_to,experience,employment,city,company_name,company_description,vacancy_count,staff_count,catalogues,source
0,Начальник отдела,Руководитель отдела сопровождения внутренней и...,"<div class=""b-b-1""> <p>Компания ""Санкт-Петербу...",2026-03-25,NaN,NaN,None,Полная занятость,Москва,Санкт-Петербургская биржа,"Компания ""Санкт-Петербургская биржа"" Вам предс...",None,None,None,scrapping
1,Java,Java (Senior)( ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМ...,"<div class=""b-b-1""> <p>Компания ""ТЕХНОЛОГИИ ОТ...",2026-03-27,NaN,NaN,Более 6 лет,Полная занятость,Москва,ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМАЦИИ,"Компания ""ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМАЦИИ""...",None,None,None,scrapping
2,Специалист,Middle+ AQA специалист,"<div class=""b-b-1""> <p>Компания ""Компания Рите...",2026-03-26,140000.0,180000.0,None,Полная занятость,Барнаул,Компания Ритейл Сервис,"Компания ""Компания Ритейл Сервис"" Мы - аккреди...",None,None,None,scrapping
3,Технолог,Технолог( АКРИХИН ),"<div class=""b-b-1""> <p>Компания ""АКРИХИН""</p> ...",2026-03-25,NaN,NaN,None,Полная занятость,Старая Купавна,АКРИХИН,"Компания ""АКРИХИН"" Обязанности: Обеспечивать ...",None,None,None,scrapping
4,Project manager,Project Manager Game Dev( Beresnev Games ),"<div class=""b-b-1""> <p>Компания ""Beresnev Game...",2026-03-26,NaN,NaN,None,Полная занятость,Москва,Beresnev Games,"Компания ""Beresnev Games"" Мы в поиске Project ...",None,None,None,scrapping


In [19]:
final_df.to_csv('vacancies_merged.csv', index=False, encoding='utf-8')